# 12. Does the stronger anonymization change the results?

The pipeline was first run with an anonymization made of six rules (outputs kept in `rules_only/`). Here the
regressors of step 7 on the GLiNER text are compared with the same regressors on the rules text: same outlets, folds,
settings and draws of k articles; paired bootstrap over outlets (1,000, seed 7).

In [1]:
import warnings

import numpy as np
import pandas as pd

import common as C

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
C.OUT.mkdir(parents=True, exist_ok=True)
LOG = C.OUT / "log.txt"
print(C.environment())
import boost as BO
import boost_eval as BE

art, outlets, rows = BO.load()
NEW = C.PREDICTIONS / "regression"
OLD = C.OUT / "rules_only" / "predictions" / "regression"
MODELS = ["TF-IDF", "Qwen3-8B", "Mistral-24B", "ModernBERT"]
done = [m for m in MODELS if len(list((NEW / m).glob("fold*.csv.gz"))) == 5 and len(list((OLD / m).glob("fold*.csv.gz"))) == 5]
print("compared:", done)

{'python': '3.12.3', 'seed': 42, 'split seed': 1, 'numpy': '2.5.4', 'pandas': '3.0.6', 'sklearn': '1.9.1', 'scipy': '1.18.1', 'torch': '2.5.1+cu121', 'transformers': '4.51.3', 'bitsandbytes': '0.50.2', 'trafilatura': '2.3.1'}


compared: ['TF-IDF', 'Qwen3-8B', 'Mistral-24B', 'ModernBERT']


/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [2]:
def matrix(folder, name, k):
    p = pd.concat([pd.read_csv(f) for f in sorted((folder / name).glob("fold*.csv.gz"))]).set_index("article_id").prediction
    p = p.reindex(art.article_id).to_numpy()
    draws = [d for f in range(1, 6) for d in BO.test_draws(outlets, rows, f) if d[1] == str(k)]
    t = pd.DataFrame([(d, j, p[b].mean()) for d, _, j, b in draws], columns=["domain", "draw", "prediction"])
    return t.pivot_table(index="draw", columns="domain", values="prediction").reindex(columns=outlets.index).to_numpy()


idx = np.vstack([np.arange(len(outlets))[None, :], np.random.default_rng(7).integers(0, len(outlets), (BE.B, len(outlets)))])
out = []
for m in done:
    for k in ["1", "5", "all"]:
        a = BE._one(matrix(OLD, m, k), m, k, outlets, idx)[2]
        b = BE._one(matrix(NEW, m, k), m, k, outlets, idx)[2]
        for (score, mes), v in b.items():
            d = v - a[(score, mes)]; lo, hi = np.percentile(d[1:], [2.5, 97.5])
            out.append({"model": m, "k": k, "score": score, "measure": mes, "rules": a[(score, mes)][0], "GLiNER": v[0],
                        "difference": d[0], "low": lo, "high": hi,
                        "better": bool(hi < 0) if mes in BE.LOWER else bool(lo > 0), "worse": bool(lo > 0) if mes in BE.LOWER else bool(hi < 0)})
T = pd.DataFrame(out); T.to_csv(C.OUT / "anonymization_effect.csv", index=False)

In [3]:
def table(k="all", score="stretched"):
    d = T[(T.k == k) & (T.score == score)].copy()
    f = lambda v, m: f"{v:.3f}" if m in ("Spearman", "Kendall") else f"{v:.1f}"
    d["text"] = [f"{f(a, m)} → {f(b, m)}" + (" ✓" if g else " ✗" if w else "") for a, b, m, g, w in
                 zip(d["rules"], d["GLiNER"], d.measure, d.better, d.worse)]
    return d.pivot_table(index="model", columns="measure", values="text", aggfunc="first")[BE.MEASURES].reindex(done)


table("all", "stretched")

measure,Spearman,Kendall,MAE,within 20
model,,,,
TF-IDF,0.696 → 0.675 ✗,0.518 → 0.495 ✗,16.3 → 16.9 ✗,70.8 → 67.7
Qwen3-8B,0.726 → 0.719,0.544 → 0.539,15.3 → 15.2,75.7 → 73.9
Mistral-24B,0.735 → 0.732,0.554 → 0.554,15.2 → 15.2,75.2 → 73.5
ModernBERT,0.602 → 0.613,0.444 → 0.454,18.1 → 17.7,66.4 → 68.1


In [4]:
table("all", "normal")

measure,Spearman,Kendall,MAE,within 20
model,,,,
TF-IDF,0.691 → 0.674 ✗,0.496 → 0.480 ✗,17.7 → 17.8,61.1 → 61.5
Qwen3-8B,0.722 → 0.711,0.526 → 0.514,16.1 → 16.3,69.0 → 69.9
Mistral-24B,0.735 → 0.735,0.538 → 0.540,15.7 → 15.8,68.6 → 68.1
ModernBERT,0.605 → 0.610,0.430 → 0.435,17.7 → 17.5,61.9 → 63.7


In [5]:
table("5", "stretched")

measure,Spearman,Kendall,MAE,within 20
model,,,,
TF-IDF,0.609 → 0.591 ✗,0.443 → 0.427 ✗,18.5 → 19.0 ✗,65.5 → 64.4
Qwen3-8B,0.663 → 0.659,0.488 → 0.484,17.1 → 17.1,69.4 → 69.0
Mistral-24B,0.712 → 0.706,0.533 → 0.528,16.0 → 16.2,71.4 → 70.9
ModernBERT,0.569 → 0.574,0.416 → 0.419,19.0 → 18.9,65.0 → 64.7


In [6]:
table("1", "stretched")

measure,Spearman,Kendall,MAE,within 20
model,,,,
TF-IDF,0.420 → 0.393 ✗,0.301 → 0.281 ✗,22.4 → 23.0 ✗,57.2 → 56.1 ✗
Qwen3-8B,0.510 → 0.505,0.366 → 0.362,20.8 → 21.0,60.5 → 60.6
Mistral-24B,0.627 → 0.615 ✗,0.459 → 0.451 ✗,18.1 → 18.3,66.2 → 66.4
ModernBERT,0.453 → 0.457,0.328 → 0.328,21.5 → 21.7,58.3 → 58.9
